# Medical Insurance Cost Prediction

## Business Problem

Medical insurance companies need to estimate future healthcare costs accurately. Better cost prediction helps insurers:

- Set fairer premiums
- Identify high-risk customer groups
- Manage financial risk
- Design prevention and wellness programs
- Support underwriting decisions

This project analyzes customer demographic and health-related variables, then builds machine learning models to predict medical insurance charges.

## Project Objective

The objective is not only to predict charges, but also to explain which customer factors drive higher insurance costs and provide actionable business recommendations.


## 1. Import Libraries

This section loads the Python libraries needed for data analysis, visualization, feature engineering, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)


## 2. Load Dataset

### Business Question

What customer data do we have available for predicting insurance charges?

The dataset contains customer attributes such as age, sex, BMI, number of children, smoking status, region, and insurance charges.


In [ ]:
df = pd.read_csv("insurance.csv")
df.head()


## 3. Dataset Overview

### Business Question

Is the dataset suitable for analysis and machine learning?

In this section, we check:

- Number of rows and columns
- Data types
- Summary statistics
- Missing values
- Duplicate records


In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df.describe()


### Interpretation

Use this section after running the code.

The dataset contains customer-level records with one target variable, `charges`. The key predictors include demographic, lifestyle, and location variables. If missing values are zero, the dataset is ready for modeling without imputation.

If duplicate rows exist, remove them before modeling so repeated records do not bias the model.


## 4. Data Cleaning

### Business Question

Can we trust the data before building a model?

The dataset is simple, but we still standardize text fields and remove duplicates. This improves consistency and avoids data quality issues during modeling.


In [ ]:
df = df.drop_duplicates()

for col in ["sex", "smoker", "region"]:
    df[col] = df[col].str.lower().str.strip()

print("Cleaned shape:", df.shape)
df.head()


## 5. Feature Engineering

### Business Question

Can we create more meaningful business features from the raw variables?

Raw variables are useful, but business-friendly groups make the analysis easier to interpret.

Created features:

- `bmi_category`: groups BMI into health categories
- `age_group`: groups customers by life stage
- `is_obese`: flags customers with BMI of 30 or above

These features help explain risk patterns and may improve model performance.


In [ ]:
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["underweight", "normal", "overweight", "obese"]
)

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 29, 39, 49, 59, 100],
    labels=["18-29", "30-39", "40-49", "50-59", "60+"]
)

df["is_obese"] = (df["bmi"] >= 30).astype(int)

df.head()


## 6. Executive KPI Summary

### Business Question

What is the overall cost profile of the customer base?

These KPIs provide a quick executive summary of the dataset.


In [ ]:
avg_charge = df["charges"].mean()
median_charge = df["charges"].median()
max_charge = df["charges"].max()
avg_bmi = df["bmi"].mean()
smoker_rate = df["smoker"].eq("yes").mean() * 100

print(f"Average insurance charge: ${avg_charge:,.2f}")
print(f"Median insurance charge: ${median_charge:,.2f}")
print(f"Highest insurance charge: ${max_charge:,.2f}")
print(f"Average BMI: {avg_bmi:.2f}")
print(f"Smoker rate: {smoker_rate:.1f}%")


### Interpretation

The difference between average and median charges indicates whether a small group of high-cost customers is pulling the average upward. If the average is much higher than the median, the cost distribution is likely right-skewed.

### Business Insight

Insurance cost management should focus on identifying high-risk groups rather than treating all customers as having similar expected costs.


## 7. Distribution of Insurance Charges

### Business Question

How are insurance charges distributed across customers?


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["charges"], bins=30)
plt.title("Distribution of Insurance Charges")
plt.xlabel("Insurance Charges")
plt.ylabel("Number of Customers")
plt.show()


### Interpretation

Insurance charges are usually right-skewed. Most customers have lower to moderate charges, while a smaller group creates high medical costs.

### Actionable Insight

The insurer should segment customers by risk profile. Pricing and prevention strategies should focus on high-cost groups because they have a large impact on total claims exposure.


## 8. Impact of Smoking on Insurance Charges

### Business Question

Do smokers have higher insurance charges than non-smokers?


In [ ]:
smoker_summary = (
    df.groupby("smoker")["charges"]
      .agg(["count", "mean", "median", "max"])
      .round(2)
)

smoker_summary


In [ ]:
plt.figure(figsize=(6,5))
df.boxplot(column="charges", by="smoker")
plt.title("Insurance Charges by Smoking Status")
plt.suptitle("")
plt.xlabel("Smoking Status")
plt.ylabel("Insurance Charges")
plt.show()


In [ ]:
smoker_mean = smoker_summary.loc["yes", "mean"]
non_smoker_mean = smoker_summary.loc["no", "mean"]
difference = smoker_mean - non_smoker_mean
ratio = smoker_mean / non_smoker_mean

print(f"Average smoker charge: ${smoker_mean:,.2f}")
print(f"Average non-smoker charge: ${non_smoker_mean:,.2f}")
print(f"Difference: ${difference:,.2f}")
print(f"Smokers cost {ratio:.2f} times more on average.")


### Interpretation

Smoking status is expected to be one of the strongest drivers of medical insurance charges. If smoker charges are several times higher than non-smoker charges, smoking should be treated as a major risk factor.

### Actionable Insight

The insurer should consider targeted smoking cessation programs, risk-based pricing, and prevention campaigns for smokers. This may reduce long-term claims while supporting healthier customer outcomes.


## 9. BMI and Insurance Charges

### Business Question

Does BMI influence insurance charges?


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["bmi"], df["charges"], alpha=0.6)
plt.title("BMI vs Insurance Charges")
plt.xlabel("BMI")
plt.ylabel("Insurance Charges")
plt.show()

print("Correlation between BMI and charges:", round(df["bmi"].corr(df["charges"]), 3))


In [ ]:
bmi_summary = (
    df.groupby("bmi_category")["charges"]
      .agg(["count", "mean", "median"])
      .round(2)
)

bmi_summary


### Interpretation

BMI may have a positive relationship with insurance charges, especially for customers classified as obese. The effect may be stronger when combined with other risk factors such as smoking.

### Actionable Insight

The insurer should offer wellness incentives, preventive health programs, and health coaching for high-BMI customers. These initiatives may help reduce future medical costs.


## 10. Age and Insurance Charges

### Business Question

Which age groups generate the highest insurance charges?


In [ ]:
age_summary = (
    df.groupby("age_group")["charges"]
      .agg(["count", "mean", "median"])
      .round(2)
)

age_summary


In [ ]:
plt.figure(figsize=(8,5))
age_summary["mean"].plot(kind="bar")
plt.title("Average Insurance Charges by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Charges")
plt.xticks(rotation=0)
plt.show()


### Interpretation

Insurance charges usually increase with age because older customers tend to have higher healthcare needs.

### Actionable Insight

Age should be included in pricing models, but it should not be used alone. Combining age with health and lifestyle indicators creates a fairer and more accurate risk assessment.


## 11. Children and Insurance Charges

### Business Question

Does the number of children affect insurance charges?


In [ ]:
children_summary = (
    df.groupby("children")["charges"]
      .agg(["count", "mean", "median"])
      .round(2)
)

children_summary


In [ ]:
plt.figure(figsize=(8,5))
children_summary["mean"].plot(kind="bar")
plt.title("Average Insurance Charges by Number of Children")
plt.xlabel("Number of Children")
plt.ylabel("Average Charges")
plt.xticks(rotation=0)
plt.show()


### Interpretation

The number of children may have a weaker relationship with insurance charges compared with smoking, BMI, and age.

### Actionable Insight

Family size should be considered in customer profiling, but pricing decisions should focus more heavily on stronger risk factors.


## 12. Regional Differences

### Business Question

Do insurance charges differ significantly by region?


In [ ]:
region_summary = (
    df.groupby("region")["charges"]
      .agg(["count", "mean", "median"])
      .round(2)
      .sort_values("mean", ascending=False)
)

region_summary


In [ ]:
plt.figure(figsize=(8,5))
region_summary["mean"].plot(kind="bar")
plt.title("Average Insurance Charges by Region")
plt.xlabel("Region")
plt.ylabel("Average Charges")
plt.xticks(rotation=45)
plt.show()


### Interpretation

If regional differences are small, location may be less important than individual risk factors such as smoking and BMI.

### Actionable Insight

The insurer should avoid over-prioritizing region in pricing strategy unless regional cost differences are large and consistent.


## 13. Model Building

### Business Question

Which model predicts insurance charges most accurately?

We compare three models:

- Linear Regression
- Decision Tree
- Random Forest

The model will be evaluated using:

- MAE: average prediction error in dollars
- RMSE: penalizes larger errors
- R²: percentage of variance explained by the model


In [ ]:
X = df.drop(columns=["charges"])
y = df["charges"]

numeric_features = ["age", "bmi", "children", "is_obese"]
categorical_features = ["sex", "smoker", "region", "bmi_category", "age_group"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append([name, mae, rmse, r2])
    trained_models[name] = pipeline

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
results_df = results_df.sort_values("RMSE")
results_df


### Interpretation

The best model is the one with the lowest RMSE and MAE, while maintaining a strong R².

If Random Forest performs best, this suggests the relationship between customer characteristics and insurance charges is non-linear.


## 14. Actual vs Predicted Charges

### Business Question

How close are the model predictions to actual insurance charges?


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)

plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title(f"Actual vs Predicted Charges: {best_model_name}")
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.show()

print("Best model:", best_model_name)


### Interpretation

Points closer to the diagonal line represent more accurate predictions. Larger deviations indicate customers where the model underestimates or overestimates charges.

### Actionable Insight

The model should be used as a decision-support tool. High-error cases should be reviewed carefully before using predictions for pricing decisions.


## 15. Feature Importance

### Business Question

Which factors are most important in predicting insurance charges?

Feature importance is available for tree-based models such as Random Forest.


In [ ]:
rf_pipeline = trained_models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
preprocessor_fitted = rf_pipeline.named_steps["preprocessor"]

feature_names = preprocessor_fitted.get_feature_names_out()
importances = rf_model.feature_importances_

feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
)

feature_importance.head(15)


In [ ]:
top_features = feature_importance.head(10).sort_values("Importance")

plt.figure(figsize=(8,5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


### Interpretation

The most important variables show which customer characteristics drive insurance charges. Smoking status, BMI, and age often appear among the strongest predictors.

### Actionable Insight

The insurer should focus risk management and prevention strategies on the strongest cost drivers rather than spreading resources evenly across all customer groups.


## 16. Final Business Insights

After running the notebook, update the numbers in this section.

### Insight 1: Smoking status is a major cost driver

Smokers show substantially higher average insurance charges than non-smokers.

Recommendation:

Develop smoking cessation incentives and adjust pricing according to validated smoking risk.

### Insight 2: BMI increases cost exposure

Customers with obesity-level BMI often show higher average charges.

Recommendation:

Offer wellness programs, health coaching, and preventive support for high-BMI customers.

### Insight 3: Age is associated with higher charges

Older customers tend to generate higher medical costs.

Recommendation:

Use age together with health and lifestyle variables for more accurate risk assessment.

### Insight 4: Regional differences appear less important

If regional averages are similar, individual risk factors matter more than location.

Recommendation:

Focus pricing strategy on customer-level risk rather than broad regional assumptions.

### Insight 5: Machine learning improves prediction quality

The best-performing model provides a data-driven way to estimate insurance charges.

Recommendation:

Use the model as a decision-support tool for underwriting and premium estimation.


## 17. Future Improvements

This project can be improved by:

- Adding medical history variables
- Including exercise, diet, and lifestyle data
- Testing XGBoost or CatBoost
- Applying hyperparameter tuning
- Deploying the model as a simple web app
- Monitoring model performance over time
- Testing fairness and bias before using the model in real pricing decisions


## 18. Portfolio Summary

This project demonstrates:

- Data cleaning
- Exploratory data analysis
- Feature engineering
- Regression modeling
- Model comparison
- Feature importance
- Business interpretation
- Actionable recommendations

This makes the project suitable for a GitHub portfolio, data analyst applications, and Upwork profile samples.
